# IPI — data exploration starter

**Runs anywhere.** The root is discovered from the notebook's location, not hardcoded,
and every section checks whether its data is present before touching it. On a fresh
clone you get §1, §2 (small panel) and §4's log; §3 and the raw HTML need the full
data directory, which is not in git.

Storage is flat files — no database — in four tiers, each wanting a different tool:

| tier | where | size | in git? | tool |
|---|---|---|---|---|
| derived indices | `data/pilot/*.csv`, `docs/*.json` | KB | yes | `pd.read_csv` |
| price panels | `data/pilot/*-prices.csv` | 0.6–85 MB | small ones only | `pd.read_csv` |
| pipeline intermediates | `data/cdx-index/*.tsv` | 1.3–5.8 GB | **no** | **duckdb** |
| raw archived pages | `data/pilot/html*/` | 86 GB | **no** | `gzip.open`, one at a time |

A full clone is ~44 MB. The other 124 GB lives only on the collection machine.

In [ ]:
import os, sys, gzip, json, re
from pathlib import Path
import numpy as np, pandas as pd

def find_root(start=None):
    """Project root = nearest ancestor holding CLAUDE.md and data/. No hardcoded paths."""
    start = Path(start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / "CLAUDE.md").exists() and (p / "data").is_dir():
            return p
    raise RuntimeError(f"project root not found above {start}")

ROOT = find_root()
os.chdir(ROOT)                      # every data path below is repo-relative
sys.path.insert(0, str(ROOT / "code"))   # so `import gigfilter` works, as in the scripts

try:
    import duckdb
except ImportError:
    duckdb = None

print("root    ", ROOT)
print("pandas  ", pd.__version__)
print("duckdb  ", duckdb.__version__ if duckdb else "NOT INSTALLED — §3 will skip")

### Preflight — what data is actually on this machine

In [ ]:
def first_present(*paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

# price panels, largest first — the big two are gitignored, the small ones are tracked
PRICES = first_present("data/pilot/balanced-prices.csv", "data/pilot/expanded-prices.csv",
                       "data/pilot/recent-prices.csv",   "data/pilot/pilot-prices.csv")
GMI    = first_present("data/cdx-index/gig-month-index.tsv")
HTML   = first_present("data/pilot/html-balanced", "data/pilot/html-recent", "data/pilot/html")
NICHE  = first_present("data/pilot/niche-assignment.csv")

def mb(p):
    if p is None: return "-"
    if p.is_dir(): return "dir"
    return f"{p.stat().st_size/1e6:,.1f} MB"

for label, p in [("price panel", PRICES), ("cdx index", GMI), ("html store", HTML), ("niches", NICHE)]:
    print(f"{label:12s} {'OK ' if p else 'ABSENT':4s} {str(p or ''):45s} {mb(p)}")

## 1. Derived indices — tiny, git-tracked, always available

In [ ]:
ipi   = pd.read_csv("data/pilot/recent-ipi.csv")                     # quarter, ipi
panel = pd.read_csv("data/pilot/panel-ipi.csv")                      # long historical panel
cats  = pd.read_csv("data/pilot/recent-category-indices-geks.csv")   # quarter x 7 categories

display(ipi.tail())
display(cats.tail())

In [ ]:
# the numbers frozen into the paper and the website
paper = json.load(open("data/pilot/paper-numbers.json"))
site  = json.load(open("docs/data.json"))
print(sorted(paper)[:20])
print({k: site[k] for k in ("generated", "cadence", "base_period", "panel_gigs")})

## 2. The price panel

One row per (gig, archived snapshot) with the three Fiverr package tiers.
`file_path` points back to the exact archived HTML the price came from — that is
the lineage hook; use it whenever a number looks wrong.

In [ ]:
px = pd.read_csv(PRICES)
print(f"{PRICES}  ->  {px.shape[0]:,} rows x {px.shape[1]} cols, "
      f"{px.memory_usage(deep=True).sum()/1e6:.0f} MB resident")
px.head(3)

In [ ]:
(px.groupby("year")
   .agg(n=("price_basic", "size"),
        median_basic=("price_basic", "median"),
        sellers=("seller", "nunique"))
   .assign(median_basic=lambda d: d.median_basic.round(2)))

## 3. Big pipeline intermediates — duckdb, not pandas

`gig-month-index.tsv` is 1.3 GB and headerless; the classified / deduped page
tables are ~5.7 GB each. duckdb scans them out-of-core in seconds where
`pd.read_csv` would exhaust memory. **Gitignored — this section needs the
collection machine.**

Gotcha: the parameterised form `read_csv(..., columns=$cols)` with
`params={"cols": {...}}` is *silently ignored*; duckdb falls back to dialect
sniffing and then fails on a headerless TSV. Render the struct inline.

In [ ]:
if GMI is None or duckdb is None:
    print("SKIP — needs data/cdx-index/gig-month-index.tsv (1.3 GB, not in git) and duckdb")
else:
    GMI_SQL = (f"read_csv('{GMI}', delim='\\t', header=false, "
               "columns={'gig_id':'VARCHAR','month':'VARCHAR',"
               "'timestamp':'VARCHAR','category':'VARCHAR'})")

    display(duckdb.sql(f"""
        SELECT category, count(*) AS snapshots, count(DISTINCT gig_id) AS gigs
        FROM {GMI_SQL}
        GROUP BY 1 ORDER BY snapshots DESC
    """).df())

In [ ]:
if GMI is None or duckdb is None:
    print("SKIP — see above")
else:
    cov = duckdb.sql(f"""
        SELECT month, count(DISTINCT gig_id) AS gigs
        FROM {GMI_SQL}
        GROUP BY 1 ORDER BY 1
    """).df()
    print(f"{len(cov)} months, {cov.month.min()}-{cov.month.max()}")
    ax = cov.set_index("month").plot(figsize=(12, 3), legend=False,
                                     title="distinct gigs archived per month")
    ax.set_xlabel("")

## 4. Raw archived pages

`data/pilot/html-{recent,balanced}/<seller>/<YYYYMMDD>_<slug>.html.gz` — 86 GB.
Never glob the tree. Enter through the price panel's `file_path`, or through the
download logs.

In [ ]:
row  = px.iloc[0]
cand = [Path(row.file_path.replace(".html", ".html.gz")), Path(row.file_path)]
path = next((p for p in cand if p.exists()), None)

if path is None:
    print(f"SKIP — HTML store not on this machine (wanted {cand[0]})")
else:
    html = (gzip.open(path, "rt", errors="replace").read() if path.suffix == ".gz"
            else path.read_text(errors="replace"))
    print(path, f"{len(html):,} chars")
    print(re.search(r"<title>(.*?)</title>", html, re.S).group(1)[:120])

In [ ]:
# the download logs are the cheap index into the HTML store — and they ARE in git
log = pd.read_csv("data/pilot/recent-download-log.tsv", sep="\t")
print(log.shape, log.status.value_counts().to_dict())
log.head(3)

## 5. Niche assignment — the 2026-08 event-study work

In [ ]:
if NICHE is None:
    print("SKIP — niche-assignment.csv not committed yet")
else:
    arr = pd.read_csv("data/pilot/niche-arrival.csv")
    asg = pd.read_csv(NICHE)
    print(asg.shape, "usable:", int(asg.usable.sum()))
    display(arr.dropna(subset=["arrival_quarter"])
               .sort_values("n_listings", ascending=False).head(10))